# PBAIH11 作業｜前處理決策與敏感度分析
**智慧醫療大數據實作與分析**　臺北醫學大學 智慧醫療學士學位學程　授課教師：王三源

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/wang3yuan/PBAIH/blob/main/homework/PBAIH11_HW.ipynb)

**個人作業**，兩週內完成（約 2–3 小時的工作量）。**期限：發布日起兩週**（確切日期與時間見 Im@TMU 公告）。

**開始前**：先「檔案 → 在雲端硬碟中儲存複本」，在自己的複本上作答。
**繳交（兩件事都要做）**：
1. 完成後「檔案 → 下載 → 下載 .ipynb」，上傳到 Im@TMU；檔名 `PBAIH11_HW_學號_姓名.ipynb`。
2. 在 Colab 右上角「共用」，把 `syw@tmu.edu.tw` 加為「**檢視者**」，再把這份複本的連結貼到下面「學生資訊」格的 `COLAB_LINK`。

**繳交前**：「執行階段 → 重新啟動並全部執行」，確認從頭到尾沒有錯誤。

### 情境
你是代謝體核心實驗室的分析助理。研究者拿到兩批資料，請你做前處理，並說明**哪些決定會影響結論**：
- 一批是 LC-MS 原始檔（實作一的 Sacurine 人類尿液，12 個 mzML）：峰偵測的參數要怎麼設？
- 一批是已經做好的特徵表（實作二的 MTBLS79，牛與羊的心臟組織，8 個批次）：QC 過濾、補值、漂移校正怎麼選，「牛與羊差異最大的特徵」會不會跟著變？
- 最後，研究者轉來一份同學的報告（**模擬資料**），請你判斷它的結論站不站得住。

### 每個人的數字都不一樣
程式會用你的**學號**產生一個隨機種子（seed）：第 1 部分指定給你的雜訊門檻、第 2 部分抽到的樣本都由它決定（第 3 部分的模擬資料所有人相同）。所以每個人的數字略有不同，但**解讀的邏輯是一樣的**。請在第一格填入自己的學號，不要用別人的學號。

### 評分標準（100 分）
| 項目 | 配分 | 評分重點 |
|---|---|---|
| 1. 參數敏感度（Sacurine） | 30 | 用指定的兩個雜訊門檻跑完偵測與對應；比較各類檔案特徵數、對應後特徵數、blank 過濾刪掉幾個；解讀「特徵數越多是不是越好」 |
| 2. 前處理決策與敏感度（MTBLS79） | 40 | 抽樣並說明為什麼保留全部 QC；選定流程 A 與 B 並寫理由；比較兩流程的前 10 個差異特徵（重疊、Jaccard、進出）；關掉／打開 QC-LOESS 的敏感度檢查；說明哪個決定影響最大、怎麼報告 |
| 3. 批判性解讀（模擬資料） | 30 | 指出至少三個問題、提出至少三個額外檢查、寫出你會怎麼向研究者報告 |
| GenAI 使用說明 | 必填（不計分） | 如實揭露不扣分；使用了卻未揭露，作業成績上限 60 分 |

> 💡 程式寫法可以參考課堂的兩個 notebook（實作一 `PBAIH11A`、實作二 `PBAIH11B`）：標示「**提供，不需修改**」的程式格都是從那裡整理來的，你只要補完 `# TODO` 那幾行（看到 `____` 就是要填的地方），再用文字回答「> 你的回答：」。
> 💡 卡住的順序：① 錯誤訊息的最後一行 → ② Colab 的 Gemini（按「解釋錯誤」，請它用中文解釋）→ ③ 課堂 notebook 的對應段落 → ④ 上課時舉手問。Gemini 給的程式要自己查證，並在最後的「GenAI 使用說明」揭露。
> 💡 圖表的文字用英文，因為 Colab 預設沒有中文字型。

## 學生資訊（必填）
填好三個欄位後執行這一格。`COLAB_LINK` 是你這份複本的分享連結（先按右上角「共用」把 `syw@tmu.edu.tw` 加為檢視者，再按「複製連結」）。

In [ ]:
STUDENT_ID = ""     # 你的學號，例如 "B812113001"（要加引號）
NAME = ""           # 你的姓名
COLAB_LINK = ""     # 這份 Colab 複本的分享連結（https://colab.research.google.com/...）

import hashlib
if not str(STUDENT_ID).strip():
    raise ValueError("請先在上面填入你的學號 STUDENT_ID（例如 \"B812113001\"），再執行這一格。")
STUDENT_ID = str(STUDENT_ID).strip()
SEED = int(hashlib.sha256(STUDENT_ID.encode("utf-8")).hexdigest()[:8], 16)   # 學號 → 固定的隨機種子
print("學號：", STUDENT_ID, "｜姓名：", NAME or "（尚未填寫）", "｜你的 seed：", SEED)
if not str(COLAB_LINK).startswith("https://colab.research.google.com/"):
    print("⚠️ 提醒：COLAB_LINK 還不是 Colab 連結；繳交前記得貼上，並把 syw@tmu.edu.tw 加為檢視者。")

## 0. 環境與資料（提供，不需修改）
這一格會：（1）在 Colab 安裝 `pyopenms==3.6.0`（約 1 分鐘）；（2）從課程 GitHub 下載兩份資料（Sacurine 裁切版 zip 約 46 MB、MTBLS79 xlsx 約 4 MB），已下載過就不會重複；（3）印出套件版本。

本作業以這些版本測試（＝2026 年秋 Colab 內建版本）：pyopenms 3.6.0、pandas 2.2.3、scipy 1.16.3、scikit-learn 1.6.1、statsmodels 0.15.0、matplotlib 3.10.0、seaborn 0.13.2。問 Gemini 時請說明你印出的版本。

> 下載失敗時：照實作一 0.1 節、實作二第 0 節的備援方式手動下載，上傳到 Colab 左側「檔案」面板後再執行一次。

In [ ]:
# 0. 安裝套件、下載資料、印出版本（提供，不需修改）
import os, sys, glob, time, zipfile, warnings, urllib.request
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    get_ipython().run_line_magic("pip", "install -q pyopenms==3.6.0")

LOCAL = os.environ.get("PBAIH11_DATA_DIR")       # 開發驗證用：本機資料夾（學生不用設定）
DATA_ROOT = LOCAL or "pbaih11_data"
os.makedirs(DATA_ROOT, exist_ok=True)

def find_files(pattern):
    return sorted(glob.glob(os.path.join(DATA_ROOT, "**", pattern), recursive=True))

# (a) Sacurine（實作一）：12 個 mzML + 樣本中繼資料
if len(find_files("*.mzML")) < 12:
    print("下載 Sacurine（約 46 MB）...")
    zip_path = os.path.join(DATA_ROOT, "sacurine_cropped.zip")
    urllib.request.urlretrieve("https://github.com/wang3yuan/PBAIH/releases/download/data-v1/sacurine_cropped.zip", zip_path)
    zipfile.ZipFile(zip_path).extractall(DATA_ROOT)
MZML = find_files("*.mzML")
SAC_META = find_files("sampleMetadata*.tsv")[0]

# (b) MTBLS79（實作二）：特徵表 xlsx；GitHub 失敗時改用 MetaboLights
XLSX_NAME = "Dataset07__SFPM.xlsx"
if not find_files(XLSX_NAME):
    print("下載 MTBLS79（約 4 MB）...")
    try:
        urllib.request.urlretrieve("https://raw.githubusercontent.com/wang3yuan/PBAIH/main/data/PBAIH11/" + XLSX_NAME,
                                   os.path.join(DATA_ROOT, XLSX_NAME))
    except Exception as e:
        print("GitHub 下載失敗（", e, "），改從 MetaboLights 下載 ...")
        urllib.request.urlretrieve("https://ftp.ebi.ac.uk/pub/databases/metabolights/studies/public/MTBLS79/FILES/" + XLSX_NAME,
                                   os.path.join(DATA_ROOT, XLSX_NAME))
XLSX = find_files(XLSX_NAME)[0]
assert len(MZML) == 12, "mzML 應該有 12 個，資料不完整"

import numpy as np, pandas as pd, scipy, sklearn, statsmodels, matplotlib, seaborn as sns
import matplotlib.pyplot as plt
import pyopenms as oms
from scipy import stats
for name, mod in [("pyopenms", oms), ("numpy", np), ("pandas", pd), ("scipy", scipy), ("scikit-learn", sklearn),
                  ("statsmodels", statsmodels), ("matplotlib", matplotlib), ("seaborn", sns)]:
    print(f"{name:13s} {mod.__version__}")
print("Python", sys.version.split()[0], "| Colab" if IN_COLAB else "| 本機")
print("資料就緒：mzML", len(MZML), "個；MTBLS79 xlsx 1 個")
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)
plt.rcParams["figure.dpi"] = 100

---
## 1. 參數敏感度：雜訊門檻（Sacurine，30 分）

**背景**：實作一用 `noise_threshold_int = 1000`。這個參數是 `MassTraceDetection` 的**雜訊門檻**：強度低於它的峰不能用來起始一條質量軌跡。門檻設低，低強度的訊號（可能是低濃度的真實代謝物，也可能是雜訊）會被收進來；設高，就只剩強訊號。

**這一題的做法**：程式依你的 seed 從兩組候選值各指定一個門檻（「低」：300、1000、2000；「高」：3000、5000、10000），用實作一的函式**各跑一次**：12 個檔的特徵偵測 → KD 對應（不另外做 RT 對齊）→ blank 過濾（比值門檻 3）。其他參數都和實作一相同（`mass_error_ppm` 10、`width_filtering` auto、`remove_single_traces` true）。

**1-1　實作一的函式（提供，不需修改）**

In [ ]:
# 1-1 實作一的函式（提供，不需修改）
sac_meta = pd.read_csv(SAC_META, sep="\t").set_index("sample_name").sort_values(["sampleType", "injectionOrder"])
sac_meta["file"] = [next(f for f in MZML if os.path.basename(f) == f"{s}.mzML") for s in sac_meta.index]

def detect_features(path, ppm=10.0, noise=1e3, remove_single_traces=True):
    """一個 mzML → FeatureMap（質量軌跡 → 洗脫峰 → 特徵）。"""
    e = oms.MSExperiment(); oms.MzMLFile().load(path, e); e.sortSpectra(True)
    mtd = oms.MassTraceDetection(); mtd.setLogType(oms.LogType.NONE)
    p = mtd.getDefaults(); p.setValue("mass_error_ppm", float(ppm)); p.setValue("noise_threshold_int", float(noise))
    mtd.setParameters(p)
    epd = oms.ElutionPeakDetection(); epd.setLogType(oms.LogType.NONE)
    p = epd.getDefaults(); p.setValue("width_filtering", "auto"); epd.setParameters(p)
    pk = epd.filterByPeakWidth(epd.detectPeaks(mtd.run(e, 0)))
    ffm = oms.FeatureFindingMetabo(); ffm.setLogType(oms.LogType.NONE)
    p = ffm.getDefaults(); p.setValue("remove_single_traces", "true" if remove_single_traces else "false")
    p.setValue("report_convex_hulls", "true"); ffm.setParameters(p)
    fm = oms.FeatureMap(); ffm.run(pk, fm); fm.setUniqueIds()
    return fm

def run_all(meta, ppm=10.0, noise=1e3):
    """對每個檔案跑 detect_features；回傳 (FeatureMap 清單, 每檔特徵數表)。"""
    t0 = time.time(); maps, rows = [], []
    for name, row in meta.iterrows():
        fm = detect_features(row["file"], ppm, noise)
        maps.append(fm); rows.append({"sample": name, "type": row["sampleType"], "n_features": fm.size()})
    print(f"noise_threshold_int = {noise:g}：12 個檔完成，{time.time()-t0:.1f} 秒")
    return maps, pd.DataFrame(rows)

def group_features(maps, names):
    """KD 對應：FeatureMap 清單 → 特徵表（列 = consensus 特徵、欄 = 樣本；沒對應到 = NaN）。"""
    cmap = oms.ConsensusMap()
    grouper = oms.FeatureGroupingAlgorithmKD(); grouper.setLogType(oms.LogType.NONE)
    grouper.group(maps, cmap)
    rows = []
    for c in cmap:
        r = {"mz": c.getMZ(), "rt": c.getRT()}
        for h in c.getFeatureList():
            r[names[h.getMapIndex()]] = h.getIntensity()
        rows.append(r)
    df = pd.DataFrame(rows).reindex(columns=["mz", "rt"] + list(names))
    return df[list(names)]

# 依你的 seed 指定兩個門檻
NOISE_LOW = [3e2, 1e3, 2e3]
NOISE_HIGH = [3e3, 5e3, 1e4]
rng1 = np.random.default_rng([SEED, 1])
NOISE_1 = float(rng1.choice(NOISE_LOW))
NOISE_2 = float(rng1.choice(NOISE_HIGH))
print(f"指定給你的兩個雜訊門檻：NOISE_1 = {NOISE_1:g}、NOISE_2 = {NOISE_2:g}")
print(sac_meta["sampleType"].value_counts().to_dict(), "（pool = pooled QC）")

**1-2　用兩個門檻各跑一次（補完兩行）**

用 `run_all(sac_meta, noise=...)` 分別跑 `NOISE_1` 和 `NOISE_2`。每次約 5–20 秒。

In [ ]:
maps_1, summary_1 = ____   # TODO：用 run_all 跑 NOISE_1
maps_2, summary_2 = ____   # TODO：用 run_all 跑 NOISE_2

<details><summary>💡 提示（先自己試，卡住再打開）</summary>

`run_all` 的第一個參數是樣本中繼資料表 `sac_meta`，雜訊門檻用關鍵字參數 `noise=`。它回傳兩個東西，所以左邊用兩個變數接。

</details>

**1-3　各類檔案的特徵數（補完一行）**

`summary_1` 每一列是一個檔：`type` 是檔案類別（blank／pool／sample）、`n_features` 是特徵數。請依 `type` 分組，算出每一類的最小值、中位數、最大值。

In [ ]:
def per_type(summary):
    return ____   # TODO：依 "type" 分組，對 "n_features" 算 ["min", "median", "max"]

display(pd.concat({f"noise {NOISE_1:g}": per_type(summary_1), f"noise {NOISE_2:g}": per_type(summary_2)}, axis=1))

<details><summary>💡 提示（先自己試，卡住再打開）</summary>

`summary.groupby("type")["n_features"]` 會把特徵數依類別分組；分組後用 `.agg([...])` 一次算好幾個統計量。

</details>

**1-4　KD 對應與 blank 過濾（補完一行）**

blank 過濾規則和實作一相同：樣本平均強度 ÷ blank 平均強度 **≥ 3 才保留**（缺值當 0；blank 完全沒偵測到的特徵，比值視為無限大，保留）。
> 執行時可能看到 `Warning: Only 3 data points for LOWESS fit…` 或 `Loading metabolite isotope model…`，可以忽略（實作一第 4 節說明過）。

In [ ]:
BLANK_RATIO = 3
names = list(sac_meta.index)
is_blank = sac_meta["sampleType"].eq("blank")

def blank_filter(X):
    samples = sac_meta.index[sac_meta["sampleType"].eq("sample")]
    blanks = sac_meta.index[is_blank]
    sample_mean = X[samples].fillna(0).mean(axis=1)
    blank_mean = X[blanks].fillna(0).mean(axis=1)
    ratio = pd.Series(np.where(blank_mean > 0, sample_mean / blank_mean.where(blank_mean > 0), np.inf), index=X.index)
    keep = ____   # TODO：比值「大於或等於」BLANK_RATIO 的特徵要保留（True/False 的 Series）
    return keep, blank_mean > 0

part1 = {}
for label, maps in [(f"noise {NOISE_1:g}", maps_1), (f"noise {NOISE_2:g}", maps_2)]:
    X = group_features(maps, names)
    keep, in_blank = blank_filter(X)
    n_det = X[sac_meta.index[~is_blank]].notna().sum(axis=1)       # 在 9 個非 blank 檔中偵測到幾次
    part1[label] = {"consensus features": len(X), "in any blank": int(in_blank.sum()),
                    "removed by blank filter": int((~keep).sum()), "kept": int(keep.sum()),
                    "detected in all 9 non-blank": int((n_det == 9).sum()),
                    "detected in only 1 file": int((X.notna().sum(axis=1) == 1).sum())}
part1 = pd.DataFrame(part1)
display(part1)

<details><summary>💡 提示（先自己試，卡住再打開）</summary>

和實作一練習 4 一樣：`ratio` 是 pandas Series，和數字比較（`>=`）就會得到 True/False 的 Series。

</details>

**1-5　解讀（寫 2–3 句）**
(a) 用你的數字說明：門檻從 `NOISE_1` 提高到 `NOISE_2`，blank、樣本、QC 的特徵數，以及對應後的特徵數各怎麼變？blank 過濾刪掉的數目又怎麼變？
(b) 「特徵數越多越好」嗎？多出來的特徵可能是什麼？如果 blank 過濾在某個門檻下刪掉的很少（甚至 0 個），代表資料很乾淨嗎？

> 你的回答：


---
## 2. 前處理決策與敏感度（MTBLS79，40 分）

**背景**：資料和實作二相同（DIMS；牛 C 與羊 S 的心臟組織，各 10 個生物樣本、在 8 個批次重複測量，另有 pooled QC）。研究問題是「**牛和羊差異最大的特徵是哪些**」。你要選一條前處理流程，並檢查換一個決定後，答案會不會變。

**2-1　讀入資料、依 seed 抽樣（提供，不需修改）**：隨機抽 80% 的**生物樣本**，**QC 全部保留**。

In [ ]:
# 2-1 讀入（同實作二第 1 節）並依 seed 抽樣（提供，不需修改）
raw = pd.read_excel(XLSX, sheet_name="data", index_col=0).drop(index="mean intensity")
meta_all = pd.read_excel(XLSX, sheet_name="meta").iloc[:len(raw), :4].copy()
meta_all.index = raw.index
meta_all["Batch"] = meta_all["Batch"].astype(int)
meta_all["Order"] = np.arange(1, len(meta_all) + 1)        # 假設：列順序 = 注入序（同實作二）
raw.columns = [f"mz{c:.5f}" for c in raw.columns]
X_raw = raw.replace(0, np.nan)                               # 0 視為缺值

rng2 = np.random.default_rng([SEED, 2])
bio = meta_all.index[meta_all["Class"] != "QC"]
picked = rng2.choice(bio, size=int(round(0.8 * len(bio))), replace=False)
keep_rows = (meta_all["Class"] == "QC") | meta_all.index.isin(picked)
X2, meta2 = X_raw[keep_rows], meta_all[keep_rows].copy()
is_qc2 = (meta2["Class"] == "QC").values

print(f"全部：{len(meta_all)} 次注入；抽樣後：{len(meta2)} 次（生物樣本 {(~is_qc2).sum()} / {len(bio)}、QC {is_qc2.sum()}）")
print("每個生物樣本（C1–C10、S1–S10）抽到的重複測量次數：",
      meta2.loc[~is_qc2, "Class2"].value_counts().sort_index().to_dict())
display(pd.crosstab(meta2["Batch"], meta2["Class"]))

**(a)** 為什麼抽樣時要把 QC **全部保留**？請從後面會用到 QC 的步驟（RSD 過濾、QC-LOESS、PQN 的參考圖譜）說明（1–2 句）。

> 你的回答：


**2-2　前處理函式（提供，不需修改）**

流程和實作二相同：QC RSD 過濾 → log2 尺度補值 → （可選）每批次 QC-LOESS → PQN → log2 ＋ Pareto。差異的排名指標：

- 每個特徵做**牛 vs 羊的 Welch t 檢定**，用 |t| 排名（|t| 越大，兩組差異相對於組內變異越明顯）。
- **先把同一個生物樣本的重複測量取平均**，再比較 10 頭牛 vs 10 頭羊。原因：同一頭動物量 6–7 次不是 6–7 個獨立樣本；直接把每次注入當獨立樣本（偽重複，pseudo-replication）會讓 t 值虛胖。
- Pareto 縮放對每個特徵只是「減平均、除以常數」，**不會改變 t 值**，所以用縮放前後的矩陣排名結果相同；縮放主要影響 PCA 這類多變量方法。

In [ ]:
# 2-2 前處理函式（提供，不需修改）
from sklearn.impute import KNNImputer
from statsmodels.nonparametric.smoothers_lowess import lowess

def qc_rsd(D, qc):
    return D[qc].std() / D[qc].mean() * 100

def qc_loess_correct(D, meta, frac=1.0):
    """每個峰、每個批次：用 QC 對注入序擬合 LOESS，原始值 ÷ 曲線 × 該峰 QC 中位數（同實作二第 4 節）。"""
    A = D.values.astype(float).copy()
    qc = (meta["Class"] == "QC").values; order = meta["Order"].values; batch = meta["Batch"].values
    target = np.median(A[qc], axis=0)
    for b in np.unique(batch):
        in_b = batch == b; in_qc = in_b & qc
        for j in range(A.shape[1]):
            y = A[in_qc, j]
            fit = lowess(y, order[in_qc], frac=frac, it=0, return_sorted=True)
            f = np.interp(order[in_b], fit[:, 0], fit[:, 1])
            f = np.where(f > 0, f, np.median(y))
            A[in_b, j] = A[in_b, j] / f * target[j]
    return pd.DataFrame(A, index=D.index, columns=D.columns)

def run_pipeline(X, meta, rsd_max=30, impute="KNN", loess=True):
    """X：0 已換成 NaN 的原始強度（樣本 × 特徵）。回傳 dict：Pareto 矩陣、保留的特徵數、處理後 QC RSD 中位數。"""
    qc = (meta["Class"] == "QC").values
    X = X.loc[:, qc_rsd(X, qc) <= rsd_max]                          # ⑤ QC RSD 過濾
    L = np.log2(X)
    if impute == "KNN":                                              # ⑥ 補值（log2 尺度）
        L = pd.DataFrame(KNNImputer(n_neighbors=5).fit_transform(L), index=L.index, columns=L.columns)
    elif impute == "min/2":
        L = L.fillna(L.min() - 1)                                    # log2 尺度：一半 = 減 1
    else:
        raise ValueError("impute 只能是 'KNN' 或 'min/2'")
    D = 2 ** L
    if loess:                                                        # ⑦ QC-LOESS 漂移校正
        D = qc_loess_correct(D, meta)
    ref = D[qc].median()                                             # ⑦ PQN
    D = D.div(D.div(ref, axis=1).median(axis=1), axis=0)
    L = np.log2(D)                                                   # ⑧ log2 + Pareto
    P = (L - L.mean()) / np.sqrt(L.std())
    return {"pareto": P, "n_features": P.shape[1], "qc_rsd_median": float(qc_rsd(D, qc).median())}

def rank_features(res, meta):
    """牛 vs 羊：同一生物樣本的重複測量先取平均，再做 Welch t 檢定；依 |t| 由大到小排序。"""
    bio = (meta["Class"] != "QC").values
    M = res["pareto"][bio].groupby(meta.loc[bio, "Class2"]).mean()   # 每個生物樣本一列
    cls = M.index.str[0]                                             # "C" 或 "S"
    t = stats.ttest_ind(M[cls == "C"], M[cls == "S"], equal_var=False).statistic
    return pd.Series(t, index=M.columns).reindex(pd.Series(np.abs(t), index=M.columns).sort_values(ascending=False).index)

def in_out_table(rank_1, rank_2, k=10, labels=("A", "B")):
    """列出只出現在其中一個前 k 名的特徵：它在另一條流程排第幾（或被 RSD 過濾掉）。"""
    top1, top2 = list(rank_1.index[:k]), list(rank_2.index[:k])
    pos1 = {f: i + 1 for i, f in enumerate(rank_1.index)}; pos2 = {f: i + 1 for i, f in enumerate(rank_2.index)}
    rows = []
    for f in [f for f in top1 if f not in top2] + [f for f in top2 if f not in top1]:
        rows.append({"feature": f, "only in top-10 of": labels[0] if f in top1 else labels[1],
                     f"rank in {labels[0]}": pos1.get(f, "filtered out"), f"rank in {labels[1]}": pos2.get(f, "filtered out")})
    return pd.DataFrame(rows)

**2-3　選定流程 A 與 B（補完兩行，並寫理由）**

| 決定 | 選項 | 對應的課堂內容 |
|---|---|---|
| `rsd_max`：QC RSD 門檻（%） | 20、30、40 | 實作二第 2 節（常見經驗值 20–30%，不是定論） |
| `impute`：補值法 | `"KNN"`、`"min/2"` | 實作二第 3 節與 3E（缺值機制） |
| `loess`：是否做 QC-LOESS | `True`、`False` | 實作二第 4 節 |

先選你認為最合理的**流程 A**，再選一個和 A **至少有一項不同**的**流程 B**（你想知道「如果換成這樣，結論會不會變」的那一項）。

In [ ]:
PIPE_A = dict(rsd_max=____, impute=____, loess=____)   # TODO：你的流程 A，例如 dict(rsd_max=30, impute="KNN", loess=True)
PIPE_B = dict(rsd_max=____, impute=____, loess=____)   # TODO：流程 B，至少一項和 A 不同

# 檢查（不用改）
for nm, p in [("PIPE_A", PIPE_A), ("PIPE_B", PIPE_B)]:
    assert p["rsd_max"] in (20, 30, 40) and p["impute"] in ("KNN", "min/2") and p["loess"] in (True, False), f"{nm} 的選項不在允許範圍"
assert PIPE_A != PIPE_B, "流程 B 至少要有一項決定和流程 A 不同"
print("流程 A：", PIPE_A); print("流程 B：", PIPE_B)

**(b)** 流程 A 的三個決定，各用一句話寫理由（可以引用實作二看到的現象，例如 QC RSD 分布、缺值集中在哪些批次、QC 的漂移）。
**(c)** 流程 B 改了哪一項？你預期它會讓前 10 名改變很多還是很少？為什麼？（先寫預期，再跑下一格）

> 你的回答：


**2-4　跑兩條流程、比較前 10 名（補完四行）**

In [ ]:
t0 = time.time()
res_A = run_pipeline(X2, meta2, **PIPE_A)
res_B = ____                                     # TODO：用 run_pipeline 跑流程 B（參考上一行）
rank_A = rank_features(res_A, meta2)
rank_B = rank_features(res_B, meta2)
print(f"兩條流程完成，{time.time()-t0:.1f} 秒；特徵數 A = {res_A['n_features']}、B = {res_B['n_features']}")

top_A, top_B = list(rank_A.index[:10]), list(rank_B.index[:10])
n_overlap = ____                                 # TODO：top_A 和 top_B 共同的特徵有幾個（集合的交集）
jaccard = ____                                   # TODO：Jaccard = 交集個數 ÷ 聯集個數
print(f"前 10 名重疊 {n_overlap} 個；Jaccard = {jaccard:.2f}")

display(pd.DataFrame({"A: feature": top_A, "A: t": rank_A[top_A].round(1).values,
                      "B: feature": top_B, "B: t": rank_B[top_B].round(1).values}, index=range(1, 11)))
display(in_out_table(rank_A, rank_B))

<details><summary>💡 提示（先自己試，卡住再打開）</summary>

- `run_pipeline(X2, meta2, **PIPE_B)`：`**` 會把 dict 展開成關鍵字參數。
- Python 集合：`set(a) & set(b)` 是交集、`set(a) | set(b)` 是聯集，`len(...)` 算個數。
- Jaccard 的範圍是 0（完全不同）到 1（完全相同）。前 10 名重疊 7 個時，聯集是 13 個，Jaccard = 7/13。

</details>

**2-5　敏感度檢查：把流程 A 的 QC-LOESS 開關反過來（補完一行）**

流程 A 有做 QC-LOESS 就關掉、沒做就打開，其他決定不變，看前 10 名變了多少。下表一起列出三條流程的整體比較：`spearman |t| vs A` 是「兩條流程都保留的特徵」上，|t| 排名的 Spearman 相關（看**整體**排名有多一致，而不只看前 10 名）。

In [ ]:
PIPE_A_FLIP = {**PIPE_A, "loess": not PIPE_A["loess"]}
res_F = ____                                     # TODO：用 run_pipeline 跑 PIPE_A_FLIP
rank_F = rank_features(res_F, meta2)

def compare_to_A(rank_x, res_x, k=10):
    shared = rank_A.index.intersection(rank_x.index)
    ov = len(set(rank_A.index[:k]) & set(rank_x.index[:k]))
    return {"n features": res_x["n_features"], "QC RSD median (%)": round(res_x["qc_rsd_median"], 1),
            "top-10 overlap with A": ov, "Jaccard with A": round(ov / len(set(rank_A.index[:k]) | set(rank_x.index[:k])), 2),
            "spearman |t| vs A": round(stats.spearmanr(rank_A[shared].abs(), rank_x[shared].abs()).statistic, 3)}

sens = pd.DataFrame({"A": compare_to_A(rank_A, res_A), "B": compare_to_A(rank_B, res_B),
                     "A-flip": compare_to_A(rank_F, res_F)}).T
print("A =", PIPE_A, "\nB =", PIPE_B, "\nA-flip =", PIPE_A_FLIP)
display(sens)
display(in_out_table(rank_A, rank_F, labels=("A", "A-flip")))

**(d)** 根據 2-4 與 2-5 的結果，用 **3–5 句**回答：
1. 你改動的決定裡，哪一個對「前 10 名」影響最大？用數字（重疊、Jaccard、Spearman）支持。
2. 進出前 10 名的特徵，是因為**被 RSD 過濾掉**，還是**排名在邊界附近互換**？這兩種情況的意義有什麼不同？
3. 如果要把「牛與羊差異最大的特徵」寫進報告，你會怎麼寫才誠實？（例如：附上哪些敏感度結果、強調哪些在各流程都穩定的特徵）

> 你的回答：


---
## 3. 批判性解讀：這份報告的結論站得住嗎？（**模擬資料**，30 分）

> ⚠️ 本部分的資料是**程式產生的模擬資料**，不是真實實驗；報告者「林同學」也是虛構的。目的在練習判讀，不代表任何真實研究。

研究者轉來林同學的報告，請你審閱。先執行下面兩格：第一格下載模擬資料（所有人同一份），第二格重現林同學的分析與圖。

### 林同學的報告（虛構）
- **設計**：20 位受試者的血漿，A 組 10 位、B 組 10 位。為了作業方便，**第 1 天（批次 1）只跑 A 組、第 2 天（批次 2）只跑 B 組**；每天穿插 5 次 pooled QC（把 20 個萃取液各取等量混合而成）。共 150 個特徵。
- **校正**：試了 5 種批次校正方法，**挑出讓 QC RSD 最低的那一種**（每個特徵、每個批次除以該批次 QC 的中位數），並以「QC RSD 大幅下降」作為校正成功的證據。
- **結果**：校正後 PCA 中 QC 緊密聚成一團，**兩個批次的 QC 完全重疊**；A 與 B 在 PC1 明顯分開；Welch t 檢定有很多特徵 p < 0.05（數字見下方輸出）。另外**只用這些顯著特徵**畫熱圖，A、B 完全分成兩群。
- **結論**：「**QC 聚攏、批次不分群，所以校正成功，組別 A 與 B 有顯著差異。**」

In [ ]:
# 3-0 下載模擬資料（提供，不需修改；所有人同一份）
SIM_NAME = "PBAIH11_HW_simulated.csv"
SIM_URL = "https://raw.githubusercontent.com/wang3yuan/PBAIH/main/data/PBAIH11/PBAIH11_HW_simulated.csv"
sim_path = (find_files(SIM_NAME) or [p for p in [SIM_NAME] if os.path.exists(p)] or [None])[0]
if sim_path is None:
    sim_path = os.path.join(DATA_ROOT, SIM_NAME)
    try:
        urllib.request.urlretrieve(SIM_URL, sim_path)
    except Exception as e:
        raise RuntimeError(f"模擬資料下載失敗（{e}）。請用瀏覽器打開 {SIM_URL}，另存成 {SIM_NAME}，"
                           "上傳到 Colab 左側「檔案」面板（放在最上層即可），再執行這一格。") from e
sim = pd.read_csv(sim_path)
info3 = sim[["Order", "Sample", "Batch", "Group", "IsQC"]].set_index("Order")
info3["Type"] = np.where(info3["IsQC"], "QC", "Sample")
feat_cols3 = [c for c in sim.columns if c.startswith("F")]
L3 = np.log2(sim.set_index("Order")[feat_cols3])            # 之後都在 log2 尺度分析
print("模擬資料：", L3.shape[0], "次注入 ×", L3.shape[1], "個特徵；欄位：", list(sim.columns[:5]), "+ F001…")

In [ ]:
# 3-1 重現林同學的分析（提供，不需修改）
from sklearn.decomposition import PCA
from statsmodels.stats.multitest import multipletests
qc3 = info3["Type"].eq("QC")

L3_corr = L3.copy()                                       # 林同學的校正：每批次扣掉該批次 QC 中位數（log 尺度的「除以」）
qc_ref = L3[qc3].median()
for b in [1, 2]:
    in_b = info3["Batch"].eq(b)
    L3_corr[in_b] = L3[in_b] - L3[in_b & qc3].median() + qc_ref

def rsd_lin(L):
    X = 2 ** L[qc3]
    return float((X.std() / X.mean() * 100).median())

A3, B3 = L3_corr[info3["Group"].eq("A")], L3_corr[info3["Group"].eq("B")]
p3 = pd.Series(stats.ttest_ind(A3, B3, equal_var=False).pvalue, index=L3.columns)
print(f"[林同學] QC RSD 中位數：校正前 {rsd_lin(L3):.1f}% → 校正後 {rsd_lin(L3_corr):.1f}%")
print(f"[林同學] Welch t 檢定 A vs B：{(p3 < 0.05).sum()} / {len(p3)} 個特徵 p < 0.05")

fig, axes = plt.subplots(1, 2, figsize=(12, 4.6))
style = {("QC", 1): ("#1A2238", "*", 140), ("QC", 2): ("#8D99AE", "*", 140), ("A", 1): ("#0F8B8D", "o", 40), ("B", 2): ("#EE6C4D", "s", 40)}
for ax, D, ttl in [(axes[0], L3, "Before correction"), (axes[1], L3_corr, "After QC-based correction")]:
    S = PCA(n_components=2).fit(D - D.mean())
    sc = S.transform(D - D.mean())
    for (g, b), (c, mk, sz) in style.items():
        m = (info3["Group"].eq(g) & info3["Batch"].eq(b)).values
        if m.any():
            ax.scatter(sc[m, 0], sc[m, 1], c=c, marker=mk, s=sz, label=f"{'QC' if g == 'QC' else 'group ' + g} (batch {b})")
    ax.set_xlabel(f"PC1 ({S.explained_variance_ratio_[0]:.0%})"); ax.set_ylabel(f"PC2 ({S.explained_variance_ratio_[1]:.0%})")
    ax.set_title(f"SIMULATED DATA - {ttl}"); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

**3-2　先看實驗設計（補完一行）**：做出「批次 × 組別」的交叉表（和實作二練習 1 一樣）。只看生物樣本或包含 QC 都可以。

In [ ]:
design3 = ____   # TODO：列 = 批次、欄 = 組別（都在 info3 裡：info3["Batch"]、info3["Group"]）
design3

<details><summary>💡 提示（先自己試，卡住再打開）</summary>

`pd.crosstab(列的 Series, 欄的 Series)`。

</details>

**(a)** 指出林同學的分析或推論中**至少三個問題**，每個問題用 1–2 句說明「為什麼這樣推論不成立」（可以對照投影片〈常見陷阱〉）。

> 你的回答：


**(b)** 提出**至少三個**你會要求做的額外檢查或補救（寫清楚「做什麼、看什麼結果、怎麼判斷」）。

> 你的回答：


**(c)** 用 **3–5 句**寫出你會怎麼向研究者報告（研究者不是資料分析專家，請用白話，但要清楚說明能下什麼結論、不能下什麼結論、下一步怎麼做）。

> 你的回答：


---
## GenAI 使用說明（必填，不計分）
依課程〈生成式 AI 使用指引〉（與「智慧醫療與機器學習基礎」相同）：
- 可以使用 GenAI（例如 Colab 的 Gemini、ChatGPT、Claude），但**要揭露**。**如實揭露不會扣分**；使用了卻未揭露，作業成績上限 60 分。
- 本作業用的是**公開資料**（Sacurine、MTBLS79）和程式產生的模擬資料，可以給 AI 看；但**不可上傳任何個人資料**。
- **不可透過網路使用中國營運的 AI 服務**（例如 DeepSeek、通義千問（Qwen）、Kimi、豆包、文心一言），不論網頁、App 或 API。
- AI 給的程式要能自己解釋每一行；AI 提供的文獻、數字要自己找到原始出處查證。

沒有使用也請寫「本次作業未使用 GenAI」。

> 本次作業是否使用 GenAI：
>
> 用了哪些工具（例如 Colab 的 Gemini、ChatGPT）：
>
> 用在哪一題、怎麼問（例如：第 2 部分請 Gemini 解釋 QC-LOESS 的 `frac` 參數是什麼意思）：
>
> 怎麼查證、有沒有修改它給的內容（例如：對照實作二第 4 節的說明與 statsmodels 文件）：